In [1]:
import numpy as np
import networkx as nx
from tqdm.auto import tqdm
import pickle
import time
import blockdynamicnetworks as bdn
import numpy as np
import pickle
import time



In [2]:


# ============================================================
# 1. GENERADOR SBM
# ============================================================

def generate_sbm(sizes, P, seed=None):

    rng = np.random.default_rng(seed)

    sizes = np.asarray(sizes, dtype=int)
    K = len(sizes)
    n = sizes.sum()

    if P.shape != (K, K):
        raise ValueError("P debe ser una matriz K x K.")

    if not np.allclose(P, P.T):
        raise ValueError("P debe ser simétrica.")

    if np.any((P < 0) | (P > 1)):
        raise ValueError("Las probabilidades deben estar entre 0 y 1.")

    z_true = np.repeat(np.arange(K), sizes)

    Y = np.zeros((n, n), dtype=np.int64)

    for i in range(n):
        for j in range(i + 1, n):

            p_ij = P[z_true[i], z_true[j]]

            Y[i, j] = rng.binomial(1, p_ij)
            Y[j, i] = Y[i, j]

    G = nx.from_numpy_array(Y)

    nx.set_node_attributes(
        G,
        {i: int(z_true[i]) for i in range(n)},
        "block"
    )

    return G, Y, z_true


# ============================================================
# 2. CONFIGURACIÓN GENERAL
# ============================================================

n = 50

sizes = [10, 5, 13, 15, 7]

K = len(sizes)

assert sum(sizes) == n


# ============================================================
# 3. MATRICES BASE DE LOS TRES PATRONES
# ============================================================

def pattern_horizontal():

    P = np.full((5, 5), 0.25)
    np.fill_diagonal(P, 0.75)

    return P


def pattern_vertical():

    P = np.full((5, 5), 0.25)

    # Conexiones fuertes entre bloques específicos
    pairs = [
        (0, 1),
        (2, 3),
        (1, 4),
        (3, 4)
    ]

    for a, b in pairs:
        P[a, b] = P[b, a] = 0.75

    np.fill_diagonal(P, 0.75)

    return P


def pattern_mixed():

    P = np.full((5, 5), 0.25)

    # Conexiones fuertes selectivas
    pairs = [
        (0, 2),
        (1, 3),
        (2, 3),
        (2, 4),
        (3, 4)
    ]

    for a, b in pairs:
        P[a, b] = P[b, a] = 0.75

    np.fill_diagonal(P, 0.75)

    return P


patterns = {
    "Horizontal": pattern_horizontal(),
    "Vertical": pattern_vertical(),
    "Mixed": pattern_mixed()
}


# ============================================================
# 4. TRANSFORMAR MATRICES A PROBABILIDADES USUALES Y EXTREMAS
# ============================================================

def transform_probabilities(P_base, p_in, p_out):

    P_new = np.empty_like(P_base, dtype=float)

    for i in range(P_base.shape[0]):
        for j in range(P_base.shape[1]):

            if i == j:
                P_new[i, j] = p_in
            else:
                # Conservamos la estructura de conexiones
                # entre bloques del patrón base.
                if P_base[i, j] > 0.25:
                    P_new[i, j] = p_in
                else:
                    P_new[i, j] = p_out

    return P_new


# ============================================================
# 5. DEFINIR LOS SEIS ESCENARIOS
# ============================================================

scenario_config = {

    "Horizontal_usual": {
        "pattern": "Horizontal",
        "p_in": 0.75,
        "p_out": 0.25,
        "seed": 101
    },

    "Vertical_usual": {
        "pattern": "Vertical",
        "p_in": 0.75,
        "p_out": 0.25,
        "seed": 102
    },

    "Mixed_usual": {
        "pattern": "Mixed",
        "p_in": 0.75,
        "p_out": 0.25,
        "seed": 103
    },

    "Horizontal_extreme": {
        "pattern": "Horizontal",
        "p_in": 0.95,
        "p_out": 0.05,
        "seed": 104
    },

    "Vertical_extreme": {
        "pattern": "Vertical",
        "p_in": 0.95,
        "p_out": 0.05,
        "seed": 105
    },

    "Mixed_extreme": {
        "pattern": "Mixed",
        "p_in": 0.95,
        "p_out": 0.05,
        "seed": 106
    }
}


# ============================================================
# 6. GENERAR LAS REDES
# ============================================================

networks = {}

for name, config in scenario_config.items():

    P_base = patterns[config["pattern"]]

    P = transform_probabilities(
        P_base,
        config["p_in"],
        config["p_out"]
    )

    G, Y, z_true = generate_sbm(
        sizes=sizes,
        P=P,
        seed=config["seed"]
    )

    networks[name] = {
        "G": G,
        "Y": Y,
        "z_true": z_true,
        "P": P,
        "sizes": sizes,
        "n": n,
        "K_true": K,
        "config": config
    }


# ============================================================
# 7. RESUMEN DE LOS ESCENARIOS
# ============================================================

for name, data in networks.items():

    G = data["G"]

    print(
        f"{name}: "
        f"n={G.number_of_nodes()}, "
        f"m={G.number_of_edges()}, "
        f"K={data['K_true']}, "
        f"densidad={nx.density(G):.4f}"
    )

Horizontal_usual: n=50, m=419, K=5, densidad=0.3420
Vertical_usual: n=50, m=643, K=5, densidad=0.5249
Mixed_usual: n=50, m=746, K=5, densidad=0.6090
Horizontal_extreme: n=50, m=293, K=5, densidad=0.2392
Vertical_extreme: n=50, m=641, K=5, densidad=0.5233
Mixed_extreme: n=50, m=814, K=5, densidad=0.6645


In [3]:
import numpy as np
from scipy.optimize import brentq
from scipy.special import gammaln
from scipy.stats import poisson

# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def rising_log(a, n):
    """Logaritmo del factorial ascendente (a)_n."""
    return gammaln(a + n) - gammaln(a)


# ============================================================
# 1. DP
# ============================================================

def expected_K_DP(n, alpha):
    i = np.arange(n)
    return np.sum(alpha / (alpha + i))


def calibrate_DP(n, K_target):
    return brentq(
        lambda alpha: expected_K_DP(n, alpha) - K_target,
        1e-8,
        1e6
    )


# ============================================================
# 2. PY
# ============================================================

def expected_K_PY(n, alpha, sigma):
    i = np.arange(n)
    return np.sum((alpha + sigma * i) / (alpha + i))


def calibrate_PY(n, K_target, sigma=0.1):

    if not (0 <= sigma < 1):
        raise ValueError("sigma debe estar entre 0 y 1.")

    lower = -sigma + 1e-8

    K_min = expected_K_PY(n, lower, sigma)
    K_max = expected_K_PY(n, 1e6, sigma)

    if not (K_min <= K_target <= K_max):
        raise ValueError(
            f"Objetivo no alcanzable para PY con sigma={sigma}. "
            f"El rango aproximado de E[K] es "
            f"[{K_min:.4f}, {K_max:.4f}]."
        )

    return brentq(
        lambda alpha: expected_K_PY(n, alpha, sigma) - K_target,
        lower,
        1e6
    )


# ============================================================
# 3. DM
# ============================================================

def expected_K_DM(n, beta, H):

    log_p_empty = (
        rising_log((H - 1) * beta, n)
        - rising_log(H * beta, n)
    )

    return H * (1 - np.exp(log_p_empty))


def calibrate_DM(n, K_target, H):

    if K_target > H:
        raise ValueError("K_target no puede superar H.")

    return brentq(
        lambda beta: expected_K_DM(n, beta, H) - K_target,
        1e-8,
        1e5
    )


# ============================================================
# 4. MFM
# ============================================================

def expected_K_MFM(n, rho, p_k):
    """
    p_k[m-1] = P(M=m)
    rho = parámetro Dirichlet por componente
    """

    p_k = np.asarray(p_k, dtype=float)
    p_k = p_k / p_k.sum()

    M_values = np.arange(1, len(p_k) + 1)
    expected = 0.0

    for M, pM in zip(M_values, p_k):

        if pM == 0:
            continue

        log_p_empty = (
            rising_log((M - 1) * rho, n)
            - rising_log(M * rho, n)
        )

        expected_conditional = M * (1 - np.exp(log_p_empty))
        expected += pM * expected_conditional

    return expected


def calibrate_MFM(n, K_target, p_k):

    max_K = len(p_k)

    if K_target > max_K:
        raise ValueError(
            "K_target supera el máximo número de componentes M."
        )

    return brentq(
        lambda rho: expected_K_MFM(n, rho, p_k) - K_target,
        1e-8,
        1e5
    )


# ============================================================
# 5. PCPP
# ============================================================

def expected_K_PCPP(n, theta, omega, tol=1e-10):
    """
    M = 1 + Poisson(theta).

    Usa la aproximación de Dirichlet que aparece en tu
    implementación C++ cuando omega > 0.
    """

    # Truncamiento de la distribución de M
    M_max = int(poisson.ppf(1 - tol, theta)) + 1
    M_values = np.arange(1, M_max + 1)

    probabilities = poisson.pmf(M_values - 1, theta)
    probabilities /= probabilities.sum()

    expected = 0.0

    for M, pM in zip(M_values, probabilities):

        if M == 1:
            expected_conditional = 1.0
            expected += pM * expected_conditional
            continue

        if omega == 0:

            log_p_empty = n * np.log((M - 1) / M)

        else:

            alpha_M = M / (omega**2) - 1 / M

            log_p_empty = (
                rising_log((M - 1) * alpha_M, n)
                - rising_log(M * alpha_M, n)
            )

        expected_conditional = M * (1 - np.exp(log_p_empty))
        expected += pM * expected_conditional

    return expected


def calibrate_PCPP(n, K_target, omega=0.0, tol=1e-10):

    def objective(theta):
        return expected_K_PCPP(n, theta, omega, tol) - K_target

    return brentq(objective, 1e-8, 1e4)


# ============================================================
# 6. GN: CALIBRACIÓN POR SIMULACIÓN
# ============================================================

def simulate_GN_partition(n, gamma, seed=None):

    rng = np.random.default_rng(seed)

    prior = bdn.PartitionPrior(
        bdn.PriorType.GN,
        gamma
    )

    labels = [0]

    for i in range(1, n):

        current = bdn.Partition(labels)
        K = len(set(labels))

        candidates = []

        # Asignar a cada comunidad existente
        for k in range(K):
            new_labels = labels.copy()

            old_label = sorted(set(labels))[k]
            new_labels.append(old_label)

            candidates.append(new_labels)

        # Crear nueva comunidad
        new_labels = labels.copy()
        new_labels.append(max(labels) + 1)
        candidates.append(new_labels)

        # Calcular probabilidades predictivas mediante
        # las probabilidades de las particiones candidatas
        log_probs = np.array([
            prior.log_probability(bdn.Partition(candidate))
            for candidate in candidates
        ])

        log_probs -= np.max(log_probs)
        probs = np.exp(log_probs)
        probs /= probs.sum()

        chosen = rng.choice(len(candidates), p=probs)
        labels = candidates[chosen]

    return len(set(labels))


def expected_K_GN_MC(n, gamma, simulations=200, seed=123):

    values = [
        simulate_GN_partition(
            n,
            gamma,
            seed=seed + s
        )
        for s in range(simulations)
    ]

    return np.mean(values)


def calibrate_GN_MC(n, K_target, simulations=200, seed=123):

    # Búsqueda sobre una grilla porque la estimación Monte Carlo
    # no es completamente suave.

    gamma_grid = np.linspace(0.1, 0.9, 11)

    results = []

    for gamma in gamma_grid:

        K_est = expected_K_GN_MC(
            n,
            gamma,
            simulations=simulations,
            seed=seed
        )

        results.append((gamma, K_est))

    best_gamma, best_K = min(
        results,
        key=lambda x: abs(x[1] - K_target)
    )

    return best_gamma, best_K, results

# ============================================================
# CONFIGURACIÓN
# ============================================================

K_target = 5
n = 50

# ---------------- DP ----------------
alpha_DP = calibrate_DP(n, K_target)

# ---------------- PY ----------------
sigma_PY = 0.01

alpha_PY = calibrate_PY(
    n,
    K_target,
    sigma=sigma_PY
)

# ---------------- DM ----------------
H_DM = 30

beta_DM = calibrate_DM(
    n,
    K_target,
    H=H_DM
)

# ---------------- MFM ----------------
# Distribución de M: P(M=m)
# Ejemplo: distribución uniforme sobre M=1,...,30

p_k_MFM = np.ones(30) / 30

rho_MFM = calibrate_MFM(
    n,
    K_target,
    p_k=p_k_MFM
)

# ---------------- GN ----------------
gamma_GN, K_GN_est, GN_results = calibrate_GN_MC(
    n,
    K_target,
    simulations=100
)

# ---------------- PCPP ----------------
omega_PCPP = 0.5

theta_PCPP = calibrate_PCPP(
    n,
    K_target,
    omega=omega_PCPP
)


# ============================================================
# RESULTADOS
# ============================================================

print("Objetivo E[K] =", K_target)

print("\nDP")
print("alpha =", alpha_DP)
print("E[K] =", expected_K_DP(n, alpha_DP))

print("\nPY")
print("alpha =", alpha_PY)
print("sigma =", sigma_PY)
print("E[K] =", expected_K_PY(n, alpha_PY, sigma_PY))

print("\nDM")
print("beta =", beta_DM)
print("H =", H_DM)
print("E[K] =", expected_K_DM(n, beta_DM, H_DM))

print("\nMFM")
print("rho =", rho_MFM)
print("E[K] =", expected_K_MFM(n, rho_MFM, p_k_MFM))

print("\nGN")
print("gamma =", gamma_GN)
print("E[K] estimado por simulación =", K_GN_est)

print("\nPCPP")
print("theta =", theta_PCPP)
print("omega =", omega_PCPP)
print("E[K] =", expected_K_PCPP(
    n, theta_PCPP, omega_PCPP
))

Objetivo E[K] = 5

DP
alpha = 1.1787275669831943
E[K] = 5.0

PY
alpha = 1.0161300150172323
sigma = 0.01
E[K] = 5.000000000000001

DM
beta = 0.04473669476337631
H = 30
E[K] = 4.999999999999563

MFM
rho = 0.1076381362769065
E[K] = 4.99999999999996

GN
gamma = 0.66
E[K] estimado por simulación = 5.45

PCPP
theta = 4.0058560574681445
omega = 0.5
E[K] = 5.000000000000011


In [4]:

def networkx_to_bdn(G_nx):

    nodes = list(G_nx.nodes())

    node_to_index = {
        node: i for i, node in enumerate(nodes)
    }

    G_bdn = bdn.Graph(len(nodes))

    for u, v in G_nx.edges():

        i = node_to_index[u]
        j = node_to_index[v]

        G_bdn.add_edge(i, j)

    return G_bdn, nodes, node_to_index


# alpha_DP
# alpha_PY, sigma_PY
# beta_DM, H_DM
# gamma_GN
# rho_MFM, p_k_MFM
# theta_PCPP, omega_PCPP


# DP
prior_DP = bdn.PartitionPrior(alpha=alpha_DP)

# PY
prior_PY = bdn.PartitionPrior(
    type=bdn.PriorType.PY,
    alpha=alpha_PY,
    sigma=sigma_PY
)

# DM
prior_DM = bdn.PartitionPrior(
    type=bdn.PriorType.DM,
    beta=beta_DM,
    H_total=int(H_DM)
)

# GN
prior_GN = bdn.PartitionPrior(
    type=bdn.PriorType.GN,
    gamma=gamma_GN
)

# MFM
prior_MFM = bdn.PartitionPrior(
    type=bdn.PriorType.MFM,
    rho=rho_MFM,
    p_k=p_k_MFM
)

# PCPP
prior_PCPP = bdn.PartitionPrior(
    type=bdn.PriorType.PCPP,
    theta=theta_PCPP,
    omega=omega_PCPP,
    pcpp_tol=0.01
)


# ============================================================
# 4. AGRUPAR LAS PRIORS Y SUS PARAMETROS
# ============================================================

priors = {

    "DP": {
        "prior": prior_DP,
        "parameters": {
            "alpha": alpha_DP
        }
    },

    "PY": {
        "prior": prior_PY,
        "parameters": {
            "alpha": alpha_PY,
            "sigma": sigma_PY
        }
    },

    "DM": {
        "prior": prior_DM,
        "parameters": {
            "beta": beta_DM,
            "H": H_DM
        }
    },

    "GN": {
        "prior": prior_GN,
        "parameters": {
            "gamma": gamma_GN
        }
    },

    "MFM": {
        "prior": prior_MFM,
        "parameters": {
            "rho": rho_MFM,
            "p_k": p_k_MFM
        }
    },

    "PCPP": {
        "prior": prior_PCPP,
        "parameters": {
            "theta": theta_PCPP,
            "omega": omega_PCPP,
            "pcpp_tol": 0.01
        }
    }
}


# ============================================================
# 5. PARAMETROS DE LA INFERENCIA
# ============================================================

ITERATIONS = 100_000

BURN_IN = 10_000

THINNING = 90

SEED = 123

A = 1.0
B = 1.0



In [ ]:

# ============================================================
# 6. EJECUTAR LAS 36 INFERENCIAS
# ============================================================

all_results = {}

total_runs = len(networks) * len(priors)



for scenario_name, scenario_data in tqdm(networks.items()):

    # --------------------------------------------------------
    # Convertir red de NetworkX a la librería C++
    # --------------------------------------------------------

    G_nx = scenario_data["G"]

    G_bdn, nodes, node_to_index = networkx_to_bdn(G_nx)

    z_true = scenario_data["z_true"]

    # Partición inicial: una única comunidad
    z0 = bdn.Partition(
        [0] * G_bdn.size()
    )


    # --------------------------------------------------------
    # Iterar sobre las seis priors
    # --------------------------------------------------------

    for prior_name, prior_info in priors.items():

        prior = prior_info["prior"]
        prior_parameters = prior_info["parameters"]

        run_name = f"{scenario_name}_{prior_name}"

        start_time = time.time()


        try:

            # 1. Modelo posterior
            model = bdn.PosteriorModel(
                G_bdn,
                prior,
                A,
                B
            )


            # 2. Sampler Split-Merge
            sampler = bdn.SplitMergeSampler(
                model,
                iterations=ITERATIONS,
                burn_in=BURN_IN,
                thinning=THINNING,
                seed=SEED
            )


            # 3. Inferencia
            sampler.run(z0)


            # 4. Resultados
            samples = sampler.samples()

            k_trace = sampler.number_of_communities()

            z_final = sampler.final_partition()

            results = sampler.results()

            log_posterior_trace = np.array(results.log_posterior_trace)


            # 5. Etiquetas de las muestras
            sample_labels = np.array([
                s.labels() for s in samples
            ])


            # 6. Tiempo de ejecución
            elapsed = time.time() - start_time


            # ------------------------------------------------
            # GUARDAR RESULTADOS
            # ------------------------------------------------

            all_results[run_name] = {

                # Identificación
                "scenario": scenario_name,
                "prior": prior_name,

                # Parámetros de la prior
                "prior_parameters": prior_parameters.copy(),

                # Información de la red
                "n": G_bdn.size(),
                "m": G_bdn.number_of_edges(),
                "density": nx.density(G_nx),

                # Información verdadera
                "K_true": len(np.unique(z_true)),
                "z_true": np.asarray(z_true),

                "P": scenario_data["P"],
                "sizes": scenario_data["sizes"],
                "adjacency": scenario_data["Y"],

                # Resultado inferencial
                "final_partition": np.array(
                    z_final.labels()
                ),

                "K_final": z_final.number_of_communities(),

                "samples": sample_labels,

                "K_trace": np.array(k_trace),

                "log_posterior": log_posterior_trace,

                "saved_samples": len(samples),

                # Diagnósticos del sampler
                "accepted_splits": sampler.accepted_splits(),
                "accepted_merges": sampler.accepted_merges(),

                "split_acceptance_rate": (
                    results.split_acceptance_rate
                ),

                "merge_acceptance_rate": (
                    results.merge_acceptance_rate
                ),

                "ESS_K": results.ess_k,

                "ESS_log_posterior": (
                    results.ess_log_posterior
                ),

                # Configuración
                "iterations": ITERATIONS,
                "burn_in": BURN_IN,
                "thinning": THINNING,
                "seed": SEED,
                "a": A,
                "b": B,

                # Tiempo
                "runtime_seconds": elapsed,

                "status": "success"
            }


            print(
                f"\n{run_name} | "
                f"K final={z_final.number_of_communities()} | "
                f"ESS K={results.ess_k:.2f} | "
                f"Tiempo={elapsed:.2f}s"
            )


        except Exception as e:

            all_results[run_name] = {

                "scenario": scenario_name,
                "prior": prior_name,
                "prior_parameters": prior_parameters.copy(),
                "status": "error",
                "error": str(e)
            }

            print(
                f"\nERROR en {run_name}: {e}"
            )




  0%|          | 0/6 [00:00<?, ?it/s]


Horizontal_usual_DP | K final=6 | ESS K=90.67 | Tiempo=60.88s

Horizontal_usual_PY | K final=6 | ESS K=81.75 | Tiempo=58.04s

Horizontal_usual_DM | K final=5 | ESS K=103.51 | Tiempo=47.15s

Horizontal_usual_GN | K final=4 | ESS K=115.03 | Tiempo=41.50s

Horizontal_usual_MFM | K final=5 | ESS K=139.86 | Tiempo=128.54s

Horizontal_usual_PCPP | K final=6 | ESS K=95.04 | Tiempo=113.21s

Vertical_usual_DP | K final=11 | ESS K=75.85 | Tiempo=64.22s

Vertical_usual_PY | K final=10 | ESS K=68.54 | Tiempo=71.66s

Vertical_usual_DM | K final=11 | ESS K=114.40 | Tiempo=69.27s

Vertical_usual_GN | K final=16 | ESS K=80.67 | Tiempo=83.02s

Vertical_usual_MFM | K final=10 | ESS K=81.56 | Tiempo=146.02s

Vertical_usual_PCPP | K final=9 | ESS K=90.93 | Tiempo=116.13s

Mixed_usual_DP | K final=8 | ESS K=112.09 | Tiempo=96.15s

Mixed_usual_PY | K final=7 | ESS K=195.54 | Tiempo=101.60s

Mixed_usual_DM | K final=7 | ESS K=171.92 | Tiempo=102.30s

Mixed_usual_GN | K final=7 | ESS K=107.42 | Tiempo=106.53

In [6]:

# ============================================================
# 7. GUARDAR RESULTADOS EN PICKLE
# ============================================================

filename = "SBM_simulation_36_runs.pkl"

with open(filename, "wb") as f:

    pickle.dump(
        all_results,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )




successful = sum(
    result["status"] == "success"
    for result in all_results.values()
)

failed = total_runs - successful

print("\n" + "=" * 50)
print("SIMULACIONES FINALIZADAS")
print("=" * 50)

print("Total de ejecuciones:", total_runs)
print("Exitosas:", successful)
print("Fallidas:", failed)
print("Archivo:", filename)


SIMULACIONES FINALIZADAS
Total de ejecuciones: 36
Exitosas: 36
Fallidas: 0
Archivo: SBM_simulation_36_runs.pkl
